In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authentication successful!")

Authentication successful!


In [ ]:
# Download the file you found using gsutil
!gsutil cp gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-00000-of-00150 data.tfrecord

import os
if os.path.exists("data.tfrecord"):
    size = os.path.getsize("data.tfrecord")
    print(f"SUCCESS: File downloaded! Size: {size / 1024 / 1024:.2f} MB")
else:
    raise FileNotFoundError("Download failed.")

Copying gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-00000-of-00150...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

\ [1 files][250.5 MiB/250.5 MiB]                                                
Operation completed over 1 objects/250.5 MiB.                                    
SUCCESS: File downloaded! Size: 250.47 MB


In [ ]:
!pip install waymo-open-dataset-tf-2-12-0 --no-deps
!pip install "protobuf<=3.20.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobu

In [ ]:
import tensorflow as tf
import math
import pandas as pd
import numpy as np
import sys
import os
from google.colab import files

# --- SETUP ---
sys.path.append("/usr/local/lib/python3.10/dist-packages")
try:
    from waymo_open_dataset.protos import scenario_pb2
except ImportError:
    import importlib
    importlib.reload(sys.modules['google.protobuf'])
    from waymo_open_dataset.protos import scenario_pb2

# --- CONFIGURATION ---
FILENAME = 'data.tfrecord'
print(f"Extracting DATA + BEHAVIORAL FLAGS from: {FILENAME}")

dataset = tf.data.TFRecordDataset(FILENAME, compression_type='')
data_rows = []

count = 0

# Loop through scenarios
for data in dataset:
    proto = scenario_pb2.Scenario()
    proto.ParseFromString(data.numpy())
    sc_id = proto.scenario_id

    # --- GET SPECIAL LISTS (The new stuff) ---
    # These lists contain the IDs of the interesting objects
    interest_ids = set(proto.objects_of_interest)
    predict_ids = set([p.track_index for p in proto.tracks_to_predict])
    sdc_id = proto.sdc_track_index

    # Loop through ALL tracks
    for i, track in enumerate(proto.tracks):

        # --- 1. EXTRACT IDENTIFIERS & FLAGS ---
        agent_id = track.id

        # NEW FLAGS: Check if this agent is in the special lists
        is_sdc = (track.id == sdc_id)
        is_interest = (track.id in interest_ids)  # Waymo says: "This agent is interacting"
        to_predict = (i in predict_ids)           # Waymo says: "Predict this agent"

        # Agent Type
        if track.object_type == 1: agent_type = "Vehicle"
        elif track.object_type == 2: agent_type = "Pedestrian"
        elif track.object_type == 3: agent_type = "Cyclist"
        else: agent_type = "Other"

        # --- 2. EXTRACT CURRENT STATE (Index 10) ---
        state_curr = track.states[10]

        if state_curr.valid:
            pos_x = state_curr.center_x
            pos_y = state_curr.center_y
            heading = state_curr.heading
            length = state_curr.length
            width = state_curr.width
            height = state_curr.height

            # Velocity
            vel_x = state_curr.velocity_x
            vel_y = state_curr.velocity_y
            speed = math.sqrt(vel_x**2 + vel_y**2)

            # Acceleration
            state_prev = track.states[9]
            if state_prev.valid:
                prev_speed = math.sqrt(state_prev.velocity_x**2 + state_prev.velocity_y**2)
                acceleration = (speed - prev_speed) / 0.1

                # Yaw Rate
                yaw_diff = heading - state_prev.heading
                if yaw_diff > math.pi: yaw_diff -= 2*math.pi
                if yaw_diff < -math.pi: yaw_diff += 2*math.pi
                yaw_rate = yaw_diff / 0.1
            else:
                acceleration = None
                yaw_rate = None
        else:
            # Missing data placeholders
            pos_x = None; pos_y = None; heading = None
            length = None; width = None; height = None
            speed = None; acceleration = None; yaw_rate = None
            vel_x = None; vel_y = None

        # --- 3. EXTRACT FUTURE TARGET (Index 60) ---
        state_fut = track.states[60]
        if state_fut.valid and state_curr.valid:
            future_dist = math.sqrt((state_fut.center_x - pos_x)**2 +
                                    (state_fut.center_y - pos_y)**2)
        else:
            future_dist = None

        # --- APPEND ---
        data_rows.append({
            'ScenarioID': sc_id,
            'AgentID': agent_id,
            'AgentType': agent_type,

            # THE NEW FLAGS
            'IsSDC': is_sdc,             # Self Driving Car?
            'IsInteractive': is_interest,# Was this flagged as "Interesting"?
            'ToPredict': to_predict,     # Was this flagged for the Challenge?

            'Valid_Current': state_curr.valid,
            'Valid_Future': state_fut.valid,

            # Kinematics
            'Pos_X': pos_x,
            'Pos_Y': pos_y,
            'Speed': speed,
            'Acceleration': acceleration,
            'YawRate': yaw_rate,
            'Length': length,
            'Width': width,
            'Height': height,
            'Vel_X': vel_x,
            'Vel_Y': vel_y,
            'FutureDist_5s': future_dist
        })

    count += 1
    if count % 20 == 0:
        print(f"Processed {count} scenarios...")

    if count >= 300:
        break

# --- SAVE ---
df = pd.DataFrame(data_rows)
print(f"SUCCESS! Extracted {len(df)} rows.")
df.to_csv('waymo_final_dataset.csv', index=False)
files.download('waymo_final_dataset.csv')

Extracting DATA + BEHAVIORAL FLAGS from: data.tfrecord
Processed 20 scenarios...
Processed 40 scenarios...
Processed 60 scenarios...
Processed 80 scenarios...
Processed 100 scenarios...
Processed 120 scenarios...
Processed 140 scenarios...
Processed 160 scenarios...
Processed 180 scenarios...
Processed 200 scenarios...
Processed 220 scenarios...
Processed 240 scenarios...
Processed 260 scenarios...
Processed 280 scenarios...
SUCCESS! Extracted 18311 rows.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
import matplotlib
import uuid
import sys
import os
from google.colab import files

sys.path.append("/usr/local/lib/python3.10/dist-packages")
try:
    from waymo_open_dataset.protos import scenario_pb2
except ImportError:
    import importlib
    importlib.reload(sys.modules["google.protobuf"])
    from waymo_open_dataset.protos import scenario_pb2

# --- ANOMALOUS VEHICLES FROM R OUTPUT ---
# (ScenarioID, AgentID, label)
ANOMALOUS_VEHICLES = [
    ("909244b878c6c203", {1409, 1413, 1411}, "HighSpeed_3vehicles"),
    ("5baef16beff3351",  {1554},              "HighAccel_YawRate"),
    ("c014d3c05600714f", {560},               "HighSpeed_HighAccel"),
    ("3eea09dc81191856", {998},               "YawRate_Clipping"),
    ("cffef7d3778fc611", {1265},              "HighYawRate"),
    ("308716b64a079919", {1571, 1602},        "YawRate_2vehicles"),
    ("2f8f82f1072ad710", {55},                "YawRate_Clipping"),
    ("262112a66742fe8d", {1478},              "YawRate_Clipping"),
    ("d3fd8cb194ea76f6", {1093},              "YawRate_Clipping"),
    ("2c4cec5c3ea292e2", {2861},              "YawRate_Clipping"),
    ("d4a4aac0a880c4a1", {2575},              "YawRate_Clipping"),
    ("27597bc952d2c9c7", {891},               "HighYawRate"),
    ("3e88a588e36844fc", {543},               "Truck_HighAccel"),
    ("f81f4ca1ed8c0296", {1847},              "HighSpeed_HighAccel"),
    ("fb7867184ab1a886", {142},               "HighSpeed_HighAccel"),
    ("5d3dd267ff0560f8", {46},                "Truck_ExtremeDecel"),
    ("8bdbbe8276d03a1f", {358},               "HighSpeed_HighAccel"),
]

from collections import defaultdict
scenario_to_agents = defaultdict(set)
for scenario_id, agent_ids, _ in ANOMALOUS_VEHICLES:
    scenario_to_agents[scenario_id].update(agent_ids)
TARGET_SCENARIOS = set(scenario_to_agents.keys())

# --- DATA CONVERTER ---
def scenario_to_dict(proto):
    map_points = []
    for feat in proto.map_features:
        poly = []
        if feat.HasField("lane"): poly = feat.lane.polyline
        elif feat.HasField("road_line"): poly = feat.road_line.polyline
        elif feat.HasField("road_edge"): poly = feat.road_edge.polyline
        elif feat.HasField("crosswalk"): poly = feat.crosswalk.polygon
        elif feat.HasField("speed_bump"): poly = feat.speed_bump.polygon
        for p in poly:
            map_points.append([p.x, p.y, p.z])
    if not map_points: map_points = [[0,0,0]]
    roadgraph_xyz = np.array(map_points, dtype=np.float32)
    num_agents = len(proto.tracks)
    past_x  = np.zeros((num_agents, 10), dtype=np.float32)
    past_y  = np.zeros((num_agents, 10), dtype=np.float32)
    past_valid = np.zeros((num_agents, 10), dtype=np.int64)
    curr_x  = np.zeros((num_agents, 1),  dtype=np.float32)
    curr_y  = np.zeros((num_agents, 1),  dtype=np.float32)
    curr_valid = np.zeros((num_agents, 1), dtype=np.int64)
    fut_x   = np.zeros((num_agents, 80), dtype=np.float32)
    fut_y   = np.zeros((num_agents, 80), dtype=np.float32)
    fut_valid = np.zeros((num_agents, 80), dtype=np.int64)
    for i, track in enumerate(proto.tracks):
        for t in range(10):
            if track.states[t].valid:
                past_x[i, t] = track.states[t].center_x
                past_y[i, t] = track.states[t].center_y
                past_valid[i, t] = 1
        if track.states[10].valid:
            curr_x[i, 0] = track.states[10].center_x
            curr_y[i, 0] = track.states[10].center_y
            curr_valid[i, 0] = 1
        for t in range(80):
            if track.states[11+t].valid:
                fut_x[i, t] = track.states[11+t].center_x
                fut_y[i, t] = track.states[11+t].center_y
                fut_valid[i, t] = 1
    return {
        "roadgraph_samples/xyz": roadgraph_xyz,
        "state/past/x": past_x,   "state/past/y": past_y,   "state/past/valid": past_valid,
        "state/current/x": curr_x, "state/current/y": curr_y, "state/current/valid": curr_valid,
        "state/future/x": fut_x,   "state/future/y": fut_y,   "state/future/valid": fut_valid,
    }

# --- PLOTTING ---
def create_figure_and_axes(size_pixels):
    fig, ax = plt.subplots(1, 1, num=uuid.uuid4())
    dpi = 100
    fig.set_size_inches([size_pixels/dpi, size_pixels/dpi])
    fig.set_dpi(dpi)
    fig.set_facecolor("white")
    ax.set_facecolor("white")
    ax.axis("off")
    fig.set_tight_layout(True)
    return fig, ax

def fig_canvas_image(fig):
    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    return data.reshape((h, w, 4))[:, :, :3]

def get_highlighted_colormap(proto, anomalous_ids):
    colors = np.full((len(proto.tracks), 4), [0.6, 0.6, 0.6, 0.4])
    for i, track in enumerate(proto.tracks):
        if track.id in anomalous_ids:
            colors[i] = [1.0, 0.0, 0.0, 1.0]
    return colors

def get_viewport(all_states, mask):
    valid = all_states[mask > 0]
    if valid.shape[0] == 0: return 0, 0, 100
    cy = (np.max(valid[...,1]) + np.min(valid[...,1])) / 2
    cx = (np.max(valid[...,0]) + np.min(valid[...,0])) / 2
    w  = max(np.ptp(valid[...,1]), np.ptp(valid[...,0]))
    return cy, cx, w

def visualize_one_step(states, mask, roadgraph, title, cy, cx, width, color_map, size_pixels=600):
    fig, ax = create_figure_and_axes(size_pixels)
    if len(roadgraph) > 0:
        rg = roadgraph[:, :2].T
        ax.plot(rg[0], rg[1], "k.", alpha=0.3, ms=1)
    mb = mask > 0
    ax.scatter(states[:,0][mb], states[:,1][mb], marker="o", linewidths=3, color=color_map[mb])
    ax.set_title(title, fontsize=10)
    s = max(10, width * 1.2)
    ax.axis([cx-s/2, cx+s/2, cy-s/2, cy+s/2])
    ax.set_aspect("equal")
    img = fig_canvas_image(fig)
    plt.close(fig)
    return img

def generate_frames(data, color_map):
    px, py = data["state/past/x"],    data["state/past/y"]
    cx, cy = data["state/current/x"], data["state/current/y"]
    fx, fy = data["state/future/x"],  data["state/future/y"]
    pm, cm_, fm = data["state/past/valid"], data["state/current/valid"], data["state/future/valid"]
    all_x = np.concatenate([px, cx, fx], axis=1)
    all_y = np.concatenate([py, cy, fy], axis=1)
    all_mask = np.concatenate([pm, cm_, fm], axis=1)
    vcy, vcx, vw = get_viewport(np.stack([all_x, all_y], axis=-1), all_mask)
    rg = data["roadgraph_samples/xyz"]
    imgs = []
    for i in range(10):
        s = np.stack([px[:,i], py[:,i]], -1)
        imgs.append(visualize_one_step(s, pm[:,i], rg, f"History: -{(10-i)*0.1:.1f}s", vcy, vcx, vw, color_map))
    for i in range(80):
        s = np.stack([fx[:,i], fy[:,i]], -1)
        imgs.append(visualize_one_step(s, fm[:,i], rg, f"Future: +{(i+1)*0.1:.1f}s", vcy, vcx, vw, color_map))
    return imgs

def render_and_download(proto, anomalous_ids, filename):
    print(f"  Rendering {filename}...")
    color_map = get_highlighted_colormap(proto, anomalous_ids)
    frames = generate_frames(scenario_to_dict(proto), color_map)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.axis("off")
    im = ax.imshow(frames[0])
    ani = animation.FuncAnimation(fig, lambda f: [im.set_data(f)] or [im], frames=frames, interval=100)
    ani.save(filename, writer="ffmpeg", fps=10)
    plt.close()
    files.download(filename)
    print(f"  Downloaded: {filename}")

# --- EXECUTION ---
FILENAME = "data.tfrecord"

if not os.path.exists(FILENAME):
    print("Error: data.tfrecord not found. Run the download cell first.")
else:
    print(f"Scanning for {len(TARGET_SCENARIOS)} scenarios...")
    dataset = tf.data.TFRecordDataset(FILENAME, compression_type="")
    found = {}

    for raw in dataset:
        p = scenario_pb2.Scenario()
        p.ParseFromString(raw.numpy())
        if p.scenario_id in TARGET_SCENARIOS:
            found[p.scenario_id] = p
            print(f"  Found {p.scenario_id} ({len(found)}/{len(TARGET_SCENARIOS)})")
        if len(found) == len(TARGET_SCENARIOS):
            break

    missing = TARGET_SCENARIOS - set(found.keys())
    if missing:
        print(f"Not found in this shard (try 00001): {missing}")

    print(f"
Rendering {len(found)} scenario videos...")
    for idx, (scenario_id, agent_ids, label) in enumerate(ANOMALOUS_VEHICLES):
        if scenario_id not in found:
            continue
        filename = f"vehicle_{idx+1:02d}_{label}_{scenario_id[:8]}.mp4"
        render_and_download(found[scenario_id], agent_ids, filename)

    print("
All done.")
